# TFR / ERD-ERS — all subjects

Every step lives in [`src/tfr_batch.py`](../src/tfr_batch.py); this notebook is only the driver.
Choosing which ICA components to remove is the one thing that needs a human, so the work is
split into three stages and that decision is written to disk once per subject:

| stage | call | interactive? | writes |
|---|---|---|---|
| 1 | `prepare_subject(subject)` | yes, once per subject | `Cache/<subj>_*`, `Figures/<subj>/ICA/` |
| 2 | `record_exclusions(subject, [...])` | manual decision | `configs/ica_exclusions.json` |
| 3 | `run_tfr_batch()` | no, all subjects at once | `TFRs/*.h5`, `Figures/<subj>/TFR/`, `Figures/<subj>/Contrasts/` |

Stage 3 reads the **cached** epochs, so the component indices you record always refer to exactly
the data ICA was fitted on. Re-running stage 3 is cheap and idempotent — rerun it after revising a
subject's exclusions. `Cache/` is rebuildable and gitignored; delete it whenever you like.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings('ignore')

# Qt backend: stage 1 needs clickable component topographies and a scrollable sources browser
import PyQt5
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'qt')

from src.tfr_batch import *

paths = project_paths()
print('project root :', paths.root)
print('subjects     :', list_subjects())
print(f"config       : {TFR_FREQS[0]}-{TFR_FREQS[-1]} Hz | "
      f"T = {TFR_N_CYCLES[0] / TFR_FREQS[0]:.2f} s constant (+/-2 Hz) | "
      f"baseline {TFR_BASELINE} | mode '{TFR_MODE}' | decim {TFR_DECIM}")
print()
review_status()

## Stage 1 — preprocess one subject and fit ICA

Set `subject_name` and run the cell. It loads and preprocesses every `<subject>_MI*.xdf` recording
(montage, average reference, 1-100 Hz + 50 Hz notch, epochs -5 to 6 s around the cue), concatenates
them, fits ICA on the result and classifies the components with ICLabel. Epochs + ICA are cached, so
re-running the cell is instant; pass `force=True` to rebuild from the raw files.

All 60 channels are kept for every subject — no per-subject bad-electrode dropping — so the TFRs
stack across subjects for group analysis. Noisy channels are dealt with here, as ICA components.

Three windows open:

- **component topographies** — click one to open its properties (spectrum, epochs image)
- **sources browser** — trials are colour-coded by condition, with a marker at each cue
- **ICLabel summary** — the full 7-class probability per component; hatched = ICLabel's suggestion

Saved to `Figures/<subject>/ICA/` regardless of what you decide.

In [ ]:
subject_name = 'AEH'   # <-- one subject at a time; see list_subjects() above

state = prepare_subject(subject_name)   # force=True to ignore the cache and rebuild

## Stage 2 — record which components to remove

`state['suggested']` is ICLabel's suggestion, and it is deliberately trigger-happy (on subject BA it
flagged 21 of 59 components where 7 were actually artifactual). Treat it as a shortlist: keep the
clear eye-blink / eye-movement / EMG components and anything with a physiologically implausible
topography, and leave borderline "muscle" components that carry real sensorimotor rhythm alone.

Edit the list below and run the cell. It merges into `configs/ica_exclusions.json`, so it is safe to
call again later to revise a subject — just rerun stage 3 for them afterwards.

In [ ]:
record_exclusions(subject_name, [], note='')

Now go back to stage 1, change `subject_name` to the next subject, and repeat. `review_status()`
below shows who is still outstanding.

In [ ]:
review_status()

## Stage 3 — TFRs and figures for every reviewed subject

For each subject: apply the recorded ICA exclusions to the cached epochs, convert to CSD, low-pass at
40 Hz, then per condition compute a multitaper TFR (4-34 Hz, `n_cycles = freqs` so the analysis
window is a constant 1 s, decimated to 50 Hz) and baseline it as a log ratio against the fixation
window.

Per subject that writes:

- `TFRs/<subject>_<event>_tfr.h5` — 4 files, for the group analysis in `Main2.ipynb`
- `Figures/<subject>/TFR/` — 12 files: `<event>_topo`, `<event>_joint`, `<event>_bands` per condition
- `Figures/<subject>/Contrasts/` — 6 files, every pairwise condition difference
- `Figures/<subject>/ICA/iclabel_final.png` — the components that were actually removed

Figures render on the `Agg` backend so a full run doesn't open ~180 Qt windows; the Qt backend is
restored when the batch finishes. A subject that fails is reported in the summary table and skipped
rather than aborting the run.

In [ ]:
results = run_tfr_batch()          # every reviewed subject
# results = run_tfr_batch(['BA'])  # or a subset, e.g. after revising its exclusions

## Group level

The cross-subject work reads the `TFRs/*.h5` files written above and lives in
[`Main2.ipynb`](Main2.ipynb): grand averages and contrasts, the cluster permutation test
(`run_cluster_test_tfr`), and the per-subject Mu / Beta topomap grids. Those readers still bucket
conditions by the legacy `ClosePalm` label, which the current recordings write as `MiddleHand`.